[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/dirac_kahler/dirac_kahler.ipynb)

# Dirac–Kähler Whitney Fields

This example computes Dirac–Kähler fields on a triangle mesh, fields that hold a scalar, a vector and a bivector part at once and are differentiated by a single derivative that acts on all of them together. In a wave of such a field the scalar part spreads like sound and the bivector part circulates like an eddy. Inside the domain the two travel independently, but where a pressure pulse meets the rim of a perforated disk, part of it turns into circulation. The same derivative splits a flow on that disk into a part that runs from sources to sinks, a part of local eddies, and a harmonic part that circulates around the hole.

The geometric product `corner_gradients * Full` contains the gradient, divergence and curl together. Restricting its input to `Even` joins scalar and stream potentials; its adjugate brings their face field back to the vertices. A velocity on the same surface splits into source-to-sink, eddy and harmonic parts, and of these only the source-to-sink part is built through the metric.

In differential-form notation, the Dirac–Kähler operator is $d+d^\dagger$ on inhomogeneous forms, with $d^\dagger$ the formal adjoint of the exterior derivative, and its square is the Hodge Laplacian, $(d+d^\dagger)^2=d\,d^\dagger+d^\dagger d$. A velocity $u$ enters as its flux form $\iota_u\mathrm{vol}$, and the decomposition reads $\iota_u\mathrm{vol}=\star d\phi+d\psi+h$.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice
import numpy as np

from numga import stack
from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.surfaces.dirac_kahler import render, scenarios
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
as_scalar, mv = mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Even = ga.gatype.even()                                                      # Even+
Full = ga.gatype.full()                                                      # Full+

np.set_printoptions(precision=4, suppress=True)

## A derivative that keeps every grade

Scalar potentials produce flows that cross their level curves; stream bivectors produce flows that follow them. Both contribute to the grade-one part of the same geometric derivative. A mixed-grade field at the vertices makes these two contributions visible together.


In [ ]:
n_disk_divisions = 9
disk_radius = 1.0

In [ ]:
disk_mesh, disk_vertices = scenarios.disk(disk_radius, n_disk_divisions)     # Mesh, Point-[vertices]
disk_edges = disk_mesh.d[0] * disk_vertices                                  # Point-[edges]
disk_sides = disk_mesh.boundary[2, 1] * disk_edges                           # [corners] Point-[faces]
disk_patches = (disk_sides[0] & disk_sides[1]) / 2                           # Patch-[faces]
disk_corner_gradients = disk_sides / (2 * disk_patches)                      # [corners] Plane+[faces]
gradient = (spdiag(disk_corner_gradients) * disk_mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]

# Multiplying the corner gradients by Full unites divergence, gradient, rotated stream gradient, and curl.
dirac = (spdiag(disk_corner_gradients) * disk_mesh.incidence[2, 0]).sum(axis=0) * Full  # (Full+ <- Full+)[faces <- vertices]

x, y = mv.x & disk_vertices, mv.y & disk_vertices                            # Scalar+[vertices], Scalar+[vertices]
scalar_potential = 0.5 * (x.squared() - y.squared()) + 0.35 * x              # Scalar+[vertices]
vertex_1form = (0.6 * x - 0.8 * y) * mv.x + (0.8 * x + 0.6 * y) * mv.y       # Plane+[vertices]
stream_bivector = (x * y - 0.35 * y) * mv.xy                                 # Bivector+[vertices]
vertex_multivector = scalar_potential + vertex_1form + stream_bivector       # Full+[vertices]
scalar_gradient = gradient(scalar_potential)                                 # Plane+[faces]
face_dirac = dirac(vertex_multivector)                                       # Full+[faces]

A staggered field instead stores the flow by its integrals along the edges and their circulation around each face.

In [ ]:
d1 = disk_mesh.d[1] * Scalar                                                 # (Scalar+ <- Scalar+)[faces <- edges]
disk_corner_mean = disk_mesh.incidence[2, 0].mean(axis=0)                    # Scalar+[faces <- vertices]

# The three Whitney maps embed 0-, 1-, and 2-cochains into Scalar+, Plane+, and Bivector+ on each face.
whitney0 = disk_corner_mean * Scalar                                              # (Scalar+ <- Scalar+)[faces <- vertices]
centroid_forms = (disk_corner_gradients[[2, 0, 1]] - disk_corner_gradients[[1, 2, 0]]) / 3  # [corners] Plane+[faces]
whitney1 = (spdiag(centroid_forms) * disk_mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]
whitney2 = spdiag(mv.xy / (mv.xy & disk_patches) * Scalar)                   # (Bivector+ <- Scalar+)[faces <- faces]

# The staggered Whitney reconstruction uses edge integrals, separately from the vertex derivative.
edge_midpoints = disk_mesh.incidence[1, 0].mean(axis=0) * disk_vertices      # Point-[edges]
mx, my = mv.x & edge_midpoints, mv.y & edge_midpoints                        # Scalar+[edges], Scalar+[edges]
midpoint_1form = (0.6 * mx - 0.8 * my) * mv.x + (0.8 * mx + 0.6 * my) * mv.y  # Plane+[edges]
edge_cochain = midpoint_1form & disk_edges                                   # Scalar+[edges]
face_cochain = d1(edge_cochain)                                              # Scalar+[faces]
whitney_multivector = whitney0(scalar_potential) + whitney1(edge_cochain) + whitney2(face_cochain)  # Full+[faces]

save_figure(
    render.draw_dirac_grades(disk_mesh, disk_vertices, vertex_multivector,
                             scalar_gradient, face_dirac, whitney_multivector),
    "dirac_kahler_grades",
);

The scalar gradient crosses the potential contours (upper left, red) and the stream gradient circulates (upper right, blue); their two saddle patterns sum to the uniform flow of `face_dirac.select[1]` (lower left). The lower right shows the Whitney reconstruction from edge integrals instead, where `dirac` differentiates a field sampled at vertices. The streaks fade downstream and their lengths share one magnitude scale across the panels.

In the notation of finite element exterior calculus, the staggered reconstruction is $W_0\phi+W_1u+W_2\omega$. Its inputs occupy vertices, edges and faces, respectively.


In [ ]:
print("Mean divergence (grade 0 of dirac):", face_dirac.select[0].batch().mean().to_array())
print("Mean curl density (grade 2 of dirac):", (face_dirac.select[2].batch().mean() & mv.xy.dual()).to_array())

## Flow around a hole

Local eddies and circulation around the hole can look similar in a small neighbourhood. Their paths reveal the difference: an eddy closes within the domain, while a harmonic loop winds around the missing region. A potential and a stream function pinned to zero at both rims account for the local flow; the remaining circulation needs different stream values on the two rims.

The flow is a velocity, an intrinsic `Point` on each face. Its stream part needs no metric: the stream function's gradient met with the patch, `Plane ^ patches`, runs along its level curves. Only the potential part needs the medium, `isotropic_medium(annulus_gradient)`, which carries the potential's gradient to the velocity it drives.


In [ ]:
inner_radius = 0.32
outer_radius = 1.0
n_annulus_rings = 15
circulation_strength = 0.65

In [ ]:
mesh, vertices, inner_ring, interior = scenarios.annulus(inner_radius, outer_radius, n_annulus_rings)
edges = mesh.d[0] * vertices                                                 # Point-[edges]
sides = mesh.boundary[2, 1] * edges                                          # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                                          # Patch-[faces]
areas = patches.norm()                                                       # Scalar+[faces]
corner_gradients = sides / (2 * patches)                                     # [corners] Plane+[faces]

annulus_gradient = (spdiag(corner_gradients) * mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]
annulus_even_gradient = (spdiag(corner_gradients) * mesh.incidence[2, 0]).sum(axis=0) * Even  # (Plane+ <- Even+)[faces <- vertices]
# An isotropic medium carries each face's gradient to the area-weighted velocity it drives, through the metric;
# its inverse reads a velocity as the gradient that drives it.
isotropic_medium = spdiag(areas * Plane.dual())                              # (Point- <- Plane+)[faces <- faces]
isotropic_resistance = spdiag(Plane.dual().inverse() / areas)                # (Plane+ <- Point-)[faces <- faces]
potential_velocity = isotropic_medium(annulus_gradient)                      # (Point- <- Scalar+)[faces <- vertices]
# Meeting a gradient with the patch gives the area-weighted velocity along its level curves, with no metric.
stream_velocity = spdiag(Plane ^ patches)(annulus_gradient)                  # (Point- <- Scalar+)[faces <- vertices]
even_velocity = isotropic_medium(annulus_even_gradient)                      # (Point- <- Even+)[faces <- vertices]
stream_laplacian = stream_velocity.adjugate()(isotropic_resistance(stream_velocity))  # (Antiscalar- <- Scalar+)[vertices <- vertices]
even_laplacian = annulus_even_gradient.adjugate()(even_velocity)             # (Even- <- Even+)[vertices <- vertices]

In [ ]:
squared_radii = vertices.norm_squared()                                      # Scalar+[vertices]
bubble = (squared_radii - inner_radius**2) * (outer_radius**2 - squared_radii)  # Scalar+[vertices]
abscissa, ordinate = mv.x & vertices, mv.y & vertices                        # Scalar+[vertices], Scalar+[vertices]
true_potential = bubble * (-2.2 * abscissa + 0.9 * ordinate)                 # Scalar+[vertices]
true_stream = bubble * (1.6 * abscissa + 1.9 * ordinate)                     # Scalar+[vertices]
# Different stream values on the two rims sustain circulation around the missing region.
pinned = SparseExtensor.selection(mv.context, inner_ring, mesh.n_vertices)
boundary_stream = ~pinned * as_scalar(np.ones(len(inner_ring))).field()      # Scalar+[vertices]
free_scalar = ~SparseExtensor.selection(mv.context, interior, mesh.n_vertices) * Scalar  # (Scalar+ <- Scalar+)[vertices <- interior]
interior_stream_laplacian = free_scalar.adjugate()(stream_laplacian(free_scalar))  # (Antiscalar- <- Scalar+)[interior <- interior]
boundary_load = -free_scalar.adjugate()(stream_laplacian(boundary_stream))   # Antiscalar-[interior]
harmonic_stream = boundary_stream + free_scalar(interior_stream_laplacian.solve(boundary_load))  # Scalar+[vertices]
true_harmonic = stream_velocity(harmonic_stream)                             # Point-[faces]
total_flow = potential_velocity(true_potential) + stream_velocity(true_stream) + circulation_strength * true_harmonic  # Point-[faces]

The decomposition starts from the total flow alone; the harmonic part is what remains once the source-to-sink and eddy parts are taken away.

In [ ]:
# On every interior vertex star the adjugate of the even gradient reads the flow's divergence in grade 0
# and its circulation in grade 2, so one solve on the even subalgebra recovers both potentials together.
free_even = ~SparseExtensor.selection(mv.context, interior, mesh.n_vertices) * Even  # (Even+ <- Even+)[vertices <- interior]
readings = free_even.adjugate()(annulus_even_gradient.adjugate()(total_flow))  # Even-[interior]
even_potential = free_even(free_even.adjugate()(even_laplacian(free_even)).solve(readings))  # Even+[vertices]
potential = even_potential.select[0]                                         # Scalar+[vertices]
stream = -(even_potential.select[2] & mv.xy.dual())                          # Scalar+[vertices]
potential_flow = even_velocity(potential)                                    # Point-[faces]
stream_flow = even_velocity(even_potential.select[2])                        # Point-[faces]
harmonic_flow = total_flow - potential_flow - stream_flow                    # Point-[faces]

save_figure(
    render.draw_hodge_decomposition(
        mesh, vertices, total_flow, potential, potential_flow, stream_flow, harmonic_flow,
    ),
    "dirac_kahler_hodge",
);

The local potentials vanish on both rims, which makes the three parts orthogonal. The upper row shows the total flow and its source-to-sink part; below, the stream part closes into local eddies (blue) and the harmonic part winds around the hole (purple). The streaks fade downstream with lengths proportional to velocity on one scale shared by all four panels.

In Hodge–de Rham notation, the boundary decomposition reads $L^2\Omega^1(\Omega)=dH^1_0(\Omega)\oplus\star dH^1_0(\Omega)\oplus\mathcal H^1(\Omega)$.


In [ ]:
potential_flow_energy = (isotropic_resistance(potential_flow) & potential_flow).sites.sum()
stream_flow_energy = (isotropic_resistance(stream_flow) & stream_flow).sites.sum()
harmonic_flow_energy = (isotropic_resistance(harmonic_flow) & harmonic_flow).sites.sum()
print("Component energies (potential, stream, harmonic):", stack((potential_flow_energy, stream_flow_energy, harmonic_flow_energy)).to_array())

## Boundary conversion of a wave

The scalar pulse starts with no stream bivector and no face-field motion. The boundary couples the two even grades: the pressure front splits around the hole, and a grade-two response grows at the rims. In the interior the two even grades decouple. This is the boundary condition supplied by the paired discrete operators, rather than an independently imposed rigid-wall acoustic condition. The cell that runs the wave takes about six seconds, most of it drawing the animation.


In [ ]:
horizontal, vertical = mv.x.dual(), mv.y.dual()                              # [] Point-, [] Point-
pulse_centre = -0.58 * horizontal + 0.18 * vertical                          # [] Point-
pulse_width = 0.15
dt = 0.00533
n_steps = 300
stride = 5
frame_duration_ms = 40

In [ ]:
# The wave keeps the Dirac–Kähler product `annulus_even_gradient`, which joins both even grades into one face 1-form.
corner_mean = mesh.incidence[2, 0].mean(axis=0)                              # Scalar+[faces <- vertices]
vertex_areas = ~corner_mean * areas                                          # Scalar+[vertices]
inverse_vertex_mass = spdiag((1 / vertex_areas) * Even.dual().inverse())     # (Even+ <- Even-)[vertices <- vertices]
# The regressive pairing changes sign on grade one; this adjugate is the energy-conserving partner.
even_divergence = inverse_vertex_mass(annulus_even_gradient.adjugate()(isotropic_medium))  # (Even+ <- Plane+)[vertices <- faces]


def propagate(even_state: Even, face_field: Plane,
              step_size: float, n_steps: int) -> Iterator[tuple[Even, Plane]]:
    """Yields the fourth-order Runge–Kutta trajectory of an even potential and a face 1-form under the Dirac–Kähler wave operators."""
    yield even_state, face_field
    for _ in range(n_steps):
        k1_even, k1_field = even_divergence(face_field), annulus_even_gradient(even_state)  # Even+[vertices], Plane+[faces]
        k2_even = even_divergence(face_field + 0.5 * step_size * k1_field)       # Even+[vertices]
        k2_field = annulus_even_gradient(even_state + 0.5 * step_size * k1_even)  # Plane+[faces]
        k3_even = even_divergence(face_field + 0.5 * step_size * k2_field)       # Even+[vertices]
        k3_field = annulus_even_gradient(even_state + 0.5 * step_size * k2_even)  # Plane+[faces]
        k4_even = even_divergence(face_field + step_size * k3_field)             # Even+[vertices]
        k4_field = annulus_even_gradient(even_state + step_size * k3_even)     # Plane+[faces]
        even_state = even_state + (step_size / 6) * (k1_even + 2 * k2_even + 2 * k3_even + k4_even)  # Even+[vertices]
        face_field = face_field + (step_size / 6) * (k1_field + 2 * k2_field + 2 * k3_field + k4_field)  # Plane+[faces]
        yield even_state, face_field

In [ ]:
envelope = (-(vertices - pulse_centre).norm_squared() / (2 * pulse_width**2)).exp()  # Scalar+[vertices]
initial_even = envelope.cast(Even)                                           # Even+[vertices]
initial_face_field = annulus_even_gradient(initial_even * 0.0)               # Plane+[faces]

sampled_frames = islice(propagate(initial_even, initial_face_field, dt, n_steps), 0, None, stride)
show_animation(
    render.animate_dirac_waves(mesh, vertices, sampled_frames),
    "dirac_kahler_waves",
    frame_duration_ms,
    n_frames=n_steps // stride + 1,
)

The outer reliefs lift the signed scalar (left, red) and stream-bivector (right, blue) amplitudes on one fixed height scale, against the undisturbed rim. The middle relief mixes red scalar, green face 1-form and blue bivector by their amplitudes. Height draws field amplitude, not a displacement of the mesh.

In [ ]:
# checks
even_gradient = (spdiag(disk_corner_gradients) * disk_mesh.incidence[2, 0]).sum(axis=0) * Even
u_corners = disk_mesh.incidence[2, 0] * vertex_1form
omega_corners = disk_mesh.incidence[2, 0] * stream_bivector
div_u = (disk_corner_gradients | u_corners).sum(axis=0)
grad_phi_plus_rot_omega = gradient(scalar_potential) + (disk_corner_gradients | omega_corners).sum(axis=0)
curl_u = (disk_corner_gradients ^ u_corners).sum(axis=0)
np.testing.assert_allclose((face_dirac.select[0] - div_u).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((face_dirac.select[1] - grad_phi_plus_rot_omega).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose(
    (face_dirac.select[1] - even_gradient(scalar_potential + stream_bivector)).kernel,
    0.0, atol=1e-14,
)
np.testing.assert_allclose((face_dirac.select[2] - curl_u).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((whitney_multivector.select[2] - face_dirac.select[2]).kernel, 0.0, atol=1e-13)

np.testing.assert_allclose((potential - true_potential).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((stream - true_stream).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((harmonic_flow - circulation_strength * true_harmonic).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((isotropic_resistance(potential_flow) & stream_flow).sites.sum().kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((isotropic_resistance(potential_flow) & harmonic_flow).sites.sum().kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((isotropic_resistance(stream_flow) & harmonic_flow).sites.sum().kernel, 0.0, atol=1e-12)
total_flow_energy = (isotropic_resistance(total_flow) & total_flow).sites.sum()
np.testing.assert_allclose((potential_flow_energy + stream_flow_energy + harmonic_flow_energy - total_flow_energy).kernel, 0.0, atol=1e-12)

energies = stack([
    ((vertex_areas * even_state.scalar_norm_squared()).sites.sum()
     + (areas * face_field.scalar_norm_squared()).sites.sum()) / 2
    for even_state, face_field in propagate(initial_even, initial_face_field, dt, n_steps)
])
np.testing.assert_allclose((energies / energies[0]).kernel, 1.0, rtol=1e-4)